<a href="https://colab.research.google.com/github/ABHINAV-ARCUS/Deep-Learning-neural-network/blob/main/exp4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications, Input, Model
import numpy as np
from sklearn.metrics import accuracy_score

# 1. Prepare Data (using MNIST as decided)
print("Loading data...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
def preprocess(data):
    data = np.stack((data,)*3, axis=-1)
    return tf.image.resize(data, (32, 32)).numpy() / 255.0
x_train, x_test = preprocess(x_train[:2000]), preprocess(x_test[:500])
y_train, y_test = y_train[:2000], y_test[:500]

# 2. Define Base Models
# A: Custom CNN
model_a = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

# B: Pre-trained ResNet50
base = applications.ResNet50(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
base.trainable = False
model_b = models.Sequential([
    base,
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

# 3. Train Individual Models (Manual Training + Pretraining)
for name, model in [("Custom_CNN", model_a), ("ResNet50", model_b)]:
    print(f"Training {name}...")
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train, epochs=2, verbose=0)

# 4. Ensemble: Build MLP Meta-Learner
# Get predictions from base models to use as inputs for the MLP
pred_a = model_a.predict(x_train, verbose=0)
pred_b = model_b.predict(x_train, verbose=0)
# Concatenate base model outputs
meta_features = np.concatenate([pred_a, pred_b], axis=1)

# Build MLP
meta_input = Input(shape=(meta_features.shape[1],))
# One more hidden layer (as requested)
x = layers.Dense(32, activation='relu')(meta_input)
final_output = layers.Dense(10, activation='softmax')(x)
mlp_model = Model(inputs=meta_input, outputs=final_output)

# 5. Train Meta-Learner
print("Training Meta-Learner MLP...")
mlp_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mlp_model.fit(meta_features, y_train, epochs=5, verbose=0)

# 6. Evaluate
test_a = model_a.predict(x_test, verbose=0)
test_b = model_b.predict(x_test, verbose=0)
meta_test = np.concatenate([test_a, test_b], axis=1)
final_preds = mlp_model.predict(meta_test, verbose=0)

acc = accuracy_score(y_test, np.argmax(final_preds, axis=1))
print(f"\nFinal Ensemble Accuracy: {acc:.4f}")

Loading data...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Training Custom_CNN...
Training ResNet50...
Training Meta-Learner MLP...

Final Ensemble Accuracy: 0.8920
